# Pairs Trading Analysis Example

This notebook demonstrates how to use the pairs trading system for statistical arbitrage.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.data_loader import DataLoader
from src.analysis.cointegration import CointegrationAnalyzer
from src.analysis.spread import MultiPairSpreadCalculator, SpreadCalculator
from src.backtesting.engine import BacktestEngine
from src.backtesting.performance import PerformanceAnalyzer

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Load Data

In [ ]:
# Initialize data loader
loader = DataLoader()

# Load financial sector data
price_data, sector_mapping = loader.get_sector_data(
    sectors=['Financials', 'Technology'],
    years=2,
    min_dollar_volume=10_000_000
)

print(f"Loaded {len(price_data.columns)} stocks")
print(f"Date range: {price_data.index[0]} to {price_data.index[-1]}")
print(f"Number of trading days: {len(price_data)}")

In [ ]:
# Display first few rows
price_data.head()

In [ ]:
# Plot some stock prices
fig, ax = plt.subplots(figsize=(14, 6))
price_data[['JPM', 'BAC', 'WFC', 'C']].plot(ax=ax)
ax.set_title('Financial Sector Stock Prices')
ax.set_ylabel('Price ($)')
plt.legend()
plt.show()

## 2. Find Cointegrated Pairs

In [ ]:
# Initialize cointegration analyzer
analyzer = CointegrationAnalyzer(
    significance_level=0.05,
    min_correlation=0.7,
    max_half_life=30,
    same_sector_only=True
)

# Find pairs
pairs = analyzer.find_pairs(price_data, sector_mapping, max_pairs=20)

print(f"Found {len(pairs)} cointegrated pairs")

In [ ]:
# Display pair summary
summary_df = analyzer.get_pair_summary(pairs)
summary_df

## 3. Analyze a Specific Pair

In [ ]:
# Select the top pair
top_pair = pairs[0]
print(f"Analyzing: {top_pair.ticker_a} - {top_pair.ticker_b}")
print(f"Hedge Ratio: {top_pair.hedge_ratio:.4f}")
print(f"P-Value: {top_pair.adf_pvalue:.4f}")
print(f"Half-Life: {top_pair.half_life:.2f} days")
print(f"Correlation: {top_pair.correlation:.3f}")

In [ ]:
# Calculate spread
spread_calc = SpreadCalculator(lookback_window=60)

spread = spread_calc.calculate_spread(
    price_data[top_pair.ticker_a],
    price_data[top_pair.ticker_b],
    top_pair.hedge_ratio
)

zscore = spread_calc.calculate_zscore(spread)

In [ ]:
# Plot spread and z-score
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Spread
ax1.plot(spread.index, spread.values, label='Spread', linewidth=1.5)
ax1.axhline(y=spread.mean(), color='green', linestyle='--', label='Mean')
ax1.fill_between(spread.index, 
                  spread.mean() - 2*spread.std(), 
                  spread.mean() + 2*spread.std(), 
                  alpha=0.2, color='gray')
ax1.set_ylabel('Spread')
ax1.set_title(f'{top_pair.ticker_a}-{top_pair.ticker_b} Spread')
ax1.legend()
ax1.grid(True)

# Z-score
ax2.plot(zscore.index, zscore.values, label='Z-Score', linewidth=1.5, color='purple')
ax2.axhline(y=2, color='red', linestyle='--', label='Entry Threshold (+2)')
ax2.axhline(y=-2, color='red', linestyle='--', label='Entry Threshold (-2)')
ax2.axhline(y=0, color='green', linestyle='-', label='Exit (0)', linewidth=2)
ax2.axhline(y=3, color='orange', linestyle=':', label='Stop Loss (±3)')
ax2.axhline(y=-3, color='orange', linestyle=':')
ax2.set_ylabel('Z-Score')
ax2.set_xlabel('Date')
ax2.set_title('Z-Score')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 4. Calculate Spreads for All Pairs

In [ ]:
# Calculate all spreads and z-scores
multi_spread_calc = MultiPairSpreadCalculator(lookback_window=60)

spreads = multi_spread_calc.calculate_all_spreads(price_data, pairs)
zscores = multi_spread_calc.calculate_all_zscores(spreads)

print(f"Calculated spreads for {len(spreads)} pairs")

In [ ]:
# Show current z-scores
current_zscores = multi_spread_calc.get_current_zscores(zscores)
current_zscores.sort_values(ascending=False)

In [ ]:
# Identify trading opportunities
opportunities = multi_spread_calc.identify_trading_opportunities(
    zscores,
    entry_threshold=2.0,
    exit_threshold=0.5
)

opportunities

## 5. Run Backtest

In [ ]:
# Initialize backtest engine
engine = BacktestEngine(
    initial_capital=100000,
    commission_per_trade=1.0,
    slippage_pct=0.0005,
    short_borrow_rate=0.02
)

# Run backtest
results = engine.run_walk_forward_backtest(
    price_data,
    pairs,
    train_period=252,
    test_period=63
)

print(f"Completed {results.num_trades} trades")

## 6. Analyze Performance

In [ ]:
# Calculate performance metrics
perf_analyzer = PerformanceAnalyzer()
metrics = perf_analyzer.calculate_all_metrics(
    results.equity_curve,
    results.daily_returns,
    results.trades
)

# Print report
perf_analyzer.print_performance_report(metrics)

In [ ]:
# Plot equity curve
fig, ax = plt.subplots(figsize=(14, 6))
results.equity_curve.plot(ax=ax, linewidth=2)
ax.set_title('Equity Curve', fontsize=14, fontweight='bold')
ax.set_ylabel('Portfolio Value ($)')
ax.set_xlabel('Date')
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Plot drawdown
running_max = results.equity_curve.expanding().max()
drawdown = (results.equity_curve - running_max) / running_max * 100

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(drawdown.index, drawdown.values, 0, alpha=0.3, color='red')
ax.plot(drawdown.index, drawdown.values, color='red', linewidth=1.5)
ax.set_title('Drawdown (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Drawdown (%)')
ax.set_xlabel('Date')
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Plot returns distribution
fig, ax = plt.subplots(figsize=(10, 6))
results.daily_returns.hist(bins=50, ax=ax, alpha=0.7, color='blue')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_title('Distribution of Daily Returns', fontsize=14, fontweight='bold')
ax.set_xlabel('Daily Return')
ax.set_ylabel('Frequency')
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Analyze by pair
pair_performance = perf_analyzer.analyze_by_pair(results.trades)
pair_performance

In [ ]:
# Plot performance by pair
fig, ax = plt.subplots(figsize=(12, 6))
pair_performance.sort_values('Total_PnL').plot(kind='barh', x='Pair', y='Total_PnL', ax=ax)
ax.set_title('P&L by Pair', fontsize=14, fontweight='bold')
ax.set_xlabel('Total P&L ($)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Summary

This notebook demonstrated:
1. Loading stock price data
2. Finding cointegrated pairs using the Engle-Granger method
3. Calculating spreads and z-scores
4. Identifying trading opportunities
5. Running a walk-forward backtest
6. Analyzing performance metrics

Next steps:
- Try different sectors or parameters
- Experiment with entry/exit thresholds
- Analyze individual trades
- Implement additional filters or risk management